In [ ]:
# Import libraries
import kagglehub
import os
import pandas as pd

In [ ]:
# Download latest version
path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")

print("Path to dataset files:", path)

In [ ]:
# Find the CSV file inside the downloaded folder
files = os.listdir(path)
print("\nFiles in dataset folder:", files)

csv_file = None
for f in files:
    if f.endswith(".csv"):
        csv_file = os.path.join(path, f)

In [ ]:
# Load dataset
df = pd.read_csv(csv_file)

In [ ]:
# Configure pandas to display full data without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Number of rows and columns
print("\nDataset shape (rows, columns):")
print(df.shape)

In [ ]:
# Column names
print("\nColumn names:")
print(df.columns)

In [ ]:
# Data types
print("\nData types of each column:")
print(df.dtypes)

In [ ]:
# Preview data
print("\nPreview of the dataset:")
print(df.head())

In [ ]:
# Check percentage of missing values for each column
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percentage': missing_percentage.values
})
missing_df = missing_df.sort_values('Missing_Percentage', ascending=False)

print("\nMissing Values Analysis:")
print(missing_df.to_string(index=False))

# Handle Missing Values

In [ ]:
# Numeric columns
numeric_cols = [
    'Temperature(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Speed(mph)'
]

# For numeric columns, fill missing values with the median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
# Categorical columns
categorical_cols = [
    'Weather_Condition', 'Wind_Direction',
    'Street', 'City', 'County', 'State',
    'Zipcode', 'Timezone'
]

# For categorical columns, fill missing values with "Unknown"
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [ ]:
# Datetime columns
datetime_cols = ['Weather_Timestamp', 'Start_Time', 'End_Time']

# For datetime columns, fill missing values with a default date (e.g., "2000-01-01 00:00:00")
for col in datetime_cols:
    df[col] = df[col].fillna("2000-01-01 00:00:00")

In [ ]:
# Sunrise/Sunset
df['Sunrise_Sunset'] = df['Sunrise_Sunset'].fillna("Unknown")

# Description
df['Description'] = df['Description'].fillna("No description")

# Split dataset into four tables.

In [ ]:
# -----------------------------
# Create Location table
# -----------------------------
location = df[['Start_Lat','Start_Lng','Street','City','County','State','Zipcode','Timezone']].drop_duplicates().reset_index(drop=True)
location['Location_ID'] = location.index + 1

# Reorder columns so Location_ID is first
location = location[['Location_ID','Start_Lat','Start_Lng','Street','City','County','State','Zipcode','Timezone']]

# Merge to assign Location_ID to accidents
df = df.merge(location, on=['Start_Lat','Start_Lng','Street','City','County','State','Zipcode','Timezone'], how='left')

In [ ]:
# -----------------------------
# Create Weather table
# -----------------------------
weather = df[['Weather_Timestamp','Temperature','Humidity','Pressure','Visibility','Wind_Direction','Wind_Speed','Weather_Condition']].drop_duplicates().reset_index(drop=True)
weather['Weather_ID'] = weather.index + 1

# Reorder columns so Weather_ID is first
weather = weather[['Weather_ID','Weather_Timestamp','Temperature','Humidity','Pressure','Visibility','Wind_Direction','Wind_Speed','Weather_Condition']]

df = df.merge(weather, on=['Weather_Timestamp','Temperature','Humidity','Pressure','Visibility','Wind_Direction','Wind_Speed','Weather_Condition'], how='left')

In [ ]:
# -----------------------------
# Create Road table
# -----------------------------
road = df[['Bump','Crossing','Junction','Railway','Roundabout','Stop','Traffic_Calming','Traffic_Signal']].drop_duplicates().reset_index(drop=True)
road['Road_ID'] = road.index + 1

# Reorder columns so Road_ID is first
road = road[['Road_ID','Bump','Crossing','Junction','Railway','Roundabout','Stop','Traffic_Calming','Traffic_Signal']]

df = df.merge(road, on=['Bump','Crossing','Junction','Railway','Roundabout','Stop','Traffic_Calming','Traffic_Signal'], how='left')

In [ ]:
# -----------------------------
# Create Accidents table
# -----------------------------
accidents = df[['ID','Severity','Start_Time','End_Time','Distance','Description','Sunrise_Sunset','Location_ID','Weather_ID','Road_ID']]

In [ ]:
# Export All Tables to CSV
location.to_csv("location.csv", index=False)
weather.to_csv("weather.csv", index=False)
# Convert boolean columns in road table to integers (1 for True, 0 for False)
road = road.replace({True: 1, False: 0})
road.to_csv("road.csv", index=False)
accidents.to_csv("accidents.csv", index=False)

print("CSV files created successfully")